# Real-Time Bitcoin Transaction Anomaly Detection using Anthropic Claude

## Introduction

This project aims to develop a real-time anomaly detection system for Bitcoin transactions using Anthropic Claude, a large language model designed with constitutional AI principles. The system will process live blockchain transaction data, detect anomalies, and generate interpretable natural language explanations for each flagged transaction.

The project follows a modular pipeline: data ingestion, feature engineering, rule-based and model-based anomaly detection, time-series pattern tracking, visualization, and system deployment.

## Objective

- Ingest real-time Bitcoin transaction data from the Blockchair API.
- Extract structural and behavioral features from transactions.
- Apply a two-stage anomaly detection system (rule-based and LLM-based).
- Generate interpretable explanations for flagged transactions using Anthropic Claude.
- Visualize the results through dashboards and integrate alerts.
- Prepare the system for scalable deployment using Spark and AWS.

## Project Workflow

1. Data Ingestion from Blockchair API
2. Feature Engineering and Time Bucketing
3. Anomaly Detection:
   - Rule-based Filtering
   - Claude LLM Analysis
4. Temporal Pattern Detection
5. Alerting and Dashboarding
6. Deployment and Auto-scaling
7. Testing and Final Reporting

## 1. Data Ingestion

We begin by retrieving live Bitcoin transaction data from the Blockchair API. In this step, we extract a clean subset of useful features and prepare them for further processing and analysis.

### 1.1 Connect to API and Retrieve Transactions

We fetch a batch of recent Bitcoin transactions using Blockchair's public API.

In [9]:
import requests
import pandas as pd

url = "https://api.blockchair.com/bitcoin/transactions?limit=10"
response = requests.get(url)

if response.status_code == 200:
    transactions = response.json()["data"]
    print(f"Retrieved {len(transactions)} transactions.")
else:
    raise Exception(f"API error {response.status_code}: {response.text}")

Retrieved 10 transactions.


### 1.2 Extract Transaction Fields

We extract the core transaction-level features needed for anomaly detection. The following key fields are extracted from the Blockchair API for further processing and anomaly detection:

| Field Name         | Description |
|--------------------|-------------|
| `tx_hash`          | Unique identifier for each Bitcoin transaction |
| `time`             | Timestamp of when the transaction occurred |
| `size`             | Size of the transaction in bytes |
| `fee_usd`          | Transaction fee in USD |
| `input_count`      | Number of input addresses in the transaction |
| `output_count`     | Number of output addresses in the transaction |
| `input_total_usd`  | Total value of all inputs in USD |
| `output_total_usd` | Total value of all outputs in USD |

These features are chosen for their high signal relevance in detecting anomalies in transaction behavior.

In [14]:
rows = []
for tx in transactions:
    rows.append({
        "tx_hash": tx["hash"],
        "time": tx["time"],
        "size": tx["size"],
        "fee_usd": tx["fee_usd"],
        "input_count": tx["input_count"],
        "output_count": tx["output_count"],
        "input_total_usd": tx["input_total_usd"],
        "output_total_usd": tx["output_total_usd"]
    })

df = pd.DataFrame(rows)
df["time"] = pd.to_datetime(df["time"])

### 1.3 Display Clean DataFrame

We truncate long transaction hashes and display a compact view of the extracted dataset.

In [19]:
df["tx_hash_short"] = df["tx_hash"].str.slice(0, 12) + "..."

df[[
    "tx_hash_short", "time", "size", "fee_usd",
    "input_count", "output_count", "input_total_usd", "output_total_usd"
]].head()

,tx_hash_short,time,size,fee_usd,input_count,output_count,input_total_usd,output_total_usd
0,26fc637d0347...,2025-04-08 12:19:05,222,0.223138,1,2,4351.985000,4351.761700
1,93bc43f3ce8a...,2025-04-08 12:19:05,320,0.238964,1,1,0.500083,0.261119
2,7cdbd3ba3217...,2025-04-08 12:19:05,320,0.238964,1,1,0.500083,0.261119
3,61ff098e200c...,2025-04-08 12:19:05,370,0.329168,2,2,80.786290,80.457120
4,79b8b755f11d...,2025-04-08 12:19:05,205,0.243711,1,2,5.470841,5.227130


## 2. Feature Engineering

In this step, we derive new features from the original transaction data to help identify abnormal behavior. These features are designed to capture structural inconsistencies, value distortions, and temporal patterns in transactions.

### 2.1 Engineered Feature Summary

The following new features are derived from the raw fields:

| Feature Name           | Description |
|------------------------|-------------|
| `fee_to_size_ratio`    | Normalized transaction fee to size ratio |
| `input_output_ratio`   | Ratio of number of inputs to number of outputs |
| `value_diff_usd`       | Net value loss (inputs - outputs) in USD |
| `value_ratio`          | Ratio of output value to input value |
| `time_1min`            | Transaction time bucketed to 1-minute intervals |
| `time_5min`            | Transaction time bucketed to 5-minute intervals |

These features will improve anomaly detection by highlighting outliers in transaction behavior and structure.

In [24]:
import numpy as np

# Feature: fee to size ratio
df["fee_to_size_ratio"] = df["fee_usd"] / df["size"]

# Feature: input-output ratio
df["input_output_ratio"] = df["input_count"] / (df["output_count"] + 1e-5)

# Feature: absolute value difference
df["value_diff_usd"] = df["input_total_usd"] - df["output_total_usd"]

# Feature: retained value ratio
df["value_ratio"] = df["output_total_usd"] / (df["input_total_usd"] + 1e-5)

# Time buckets
df["time_1min"] = df["time"].dt.floor("min")
df["time_5min"] = df["time"].dt.floor("5min")

### 2.2 Preview Engineered Features

We display a subset of the enhanced dataset, including both original and newly engineered features.

In [27]:
df[[
    "tx_hash_short", "fee_to_size_ratio", "input_output_ratio",
    "value_diff_usd", "value_ratio", "time_1min", "time_5min"
]].head()

,tx_hash_short,fee_to_size_ratio,input_output_ratio,value_diff_usd,value_ratio,time_1min,time_5min
0,26fc637d0347...,0.001005,0.499998,0.223300,0.999949,2025-04-08 12:19:00,2025-04-08 12:15:00
1,93bc43f3ce8a...,0.000747,0.999990,0.238964,0.522141,2025-04-08 12:19:00,2025-04-08 12:15:00
2,7cdbd3ba3217...,0.000747,0.999990,0.238964,0.522141,2025-04-08 12:19:00,2025-04-08 12:15:00
3,61ff098e200c...,0.000890,0.999995,0.329170,0.995925,2025-04-08 12:19:00,2025-04-08 12:15:00
4,79b8b755f11d...,0.001189,0.499998,0.243711,0.955451,2025-04-08 12:19:00,2025-04-08 12:15:00


## 3. Anomaly Detection

We implement a two-stage anomaly detection pipeline. The first stage applies simple rule-based filters to identify obvious outliers. The second stage uses Anthropic Claude to analyze transaction details and generate natural language explanations for suspicious transactions.

### 3.1 Rule-Based Filtering

We apply a basic set of static rules to flag transactions that deviate significantly from typical behavior. These help us quickly isolate high-risk candidates for deeper LLM-based inspection.

| Rule Description                                 | Logic |
|--------------------------------------------------|-------|
| High fee-to-size ratio                          | `fee_to_size_ratio > threshold` |
| High input-output fan-in or fan-out             | `input_output_ratio > upper_threshold or < lower_threshold` |
| High value discrepancy (possible value siphon)  | `value_diff_usd > threshold` |

Note: These thresholds can be tuned based on exploratory data analysis.

In [31]:
# Define rule thresholds (adjustable based on later insights)
fee_threshold = 0.01          # High fee-to-size ratio
io_ratio_upper = 10           # Fan-in
io_ratio_lower = 0.1          # Fan-out
value_diff_threshold = 1000   # Significant value gap

# Apply rule-based flags
df["flag_high_fee_ratio"] = df["fee_to_size_ratio"] > fee_threshold
df["flag_io_ratio"] = (df["input_output_ratio"] > io_ratio_upper) | (df["input_output_ratio"] < io_ratio_lower)
df["flag_value_diff"] = df["value_diff_usd"] > value_diff_threshold

# Combine rule-based flags into a single indicator
df["rule_flagged"] = df[["flag_high_fee_ratio", "flag_io_ratio", "flag_value_diff"]].any(axis=1)

### 3.2 Preview Flagged Transactions

We display the transactions that were flagged by one or more of the rule-based filters.